In [1]:
import pandas as pd

In [2]:
df_web = pd.read_csv("Datasets_limpios/clean_web_data.csv")
df_exp = pd.read_csv("Datasets_limpios/clean_experiment_clients.csv")

In [3]:
df_web.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,169,201385055_71273495308,749567106_99161211863_557568,start,2017-04-12 20:19:36
1,169,201385055_71273495308,749567106_99161211863_557568,step_1,2017-04-12 20:19:45
2,169,201385055_71273495308,749567106_99161211863_557568,step_2,2017-04-12 20:20:31
3,169,201385055_71273495308,749567106_99161211863_557568,step_3,2017-04-12 20:22:05
4,169,201385055_71273495308,749567106_99161211863_557568,confirm,2017-04-12 20:23:09


In [4]:
df_exp.head()

,client_id,variation
0,9988021,Test
1,8320017,Test
2,4033851,Control
3,1982004,Test
4,9294070,Control


In [5]:
df_web.info()

<class 'pandas.DataFrame'>
RangeIndex: 744641 entries, 0 to 744640
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   client_id     744641 non-null  int64
 1   visitor_id    744641 non-null  str  
 2   visit_id      744641 non-null  str  
 3   process_step  744641 non-null  str  
 4   date_time     744641 non-null  str  
dtypes: int64(1), str(4)
memory usage: 28.4 MB


In [6]:
#Convertir date_time en fecha
df_web["date_time"] = pd.to_datetime(df_web["date_time"])
df_web.info()

<class 'pandas.DataFrame'>
RangeIndex: 744641 entries, 0 to 744640
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   client_id     744641 non-null  int64         
 1   visitor_id    744641 non-null  str           
 2   visit_id      744641 non-null  str           
 3   process_step  744641 non-null  str           
 4   date_time     744641 non-null  datetime64[us]
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 28.4 MB


In [8]:
#Vamos a hacer merge de web_data y experiment_clients para calcular el KPI: Completion rate
df_merge_web_exp = df_web.merge(df_exp, on="client_id", how="inner")
df_merge_web_exp

,client_id,visitor_id,visit_id,process_step,date_time,variation
0,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test
...,...,...,...,...,...,...
317230,9999729,834634258_21862004160,870243567_56915814033_814203,confirm,2017-05-08 16:09:40,Test
317231,9999729,604429154_69247391147,99583652_41711450505_426179,start,2017-04-05 13:40:49,Test
317232,9999729,604429154_69247391147,99583652_41711450505_426179,step_1,2017-04-05 13:41:04,Test
317233,9999832,145538019_54444341400,472154369_16714624241_585315,start,2017-05-16 16:46:03,Test


In [11]:
#KPI 1: Completion rate
#Formula: (Completed users/total users) * 100
#1. Calculo completed users, considerando unicos. 
completed_users = df_merge_web_exp[df_merge_web_exp["process_step"] == 'confirm'].groupby("variation")["client_id"].nunique()
completed_users

variation
Control    15434
Test       18687
Name: client_id, dtype: int64

In [12]:
#2. Calculo total users
total_users = df_merge_web_exp.groupby("variation")["client_id"].nunique()
total_users

variation
Control    23532
Test       26968
Name: client_id, dtype: int64

In [13]:
#3. Calculo del completion rate
completion_rate = ((completed_users / total_users) * 100).round(2)
completion_rate

variation
Control    65.59
Test       69.29
Name: client_id, dtype: float64

Interpretacion: Podemos ver que el completion rate del grupo Test es superior, lo cual indica que la nueva inte4rfaz podria estar ayudando a que más usuarios completen el proceso hasta el final.

In [ ]:
#KPI 2: Time Spent on Each Step
#Calcular el tiempo que se tarda en pasar de un step a otro para cada cliente_id y visit_id. Si vemos que los tiempos son muy elevados, ya probamos otras opciones. 
#Formula: Time Step on Each Step = Next Step Timestamp - Current Step Timestamp

In [15]:
#1. Primero vamos a ordenar el merged dataframe.
df_merge_web_exp = df_merge_web_exp.sort_values(by=["client_id", "visit_id", "date_time"]).reset_index(drop=True)
df_merge_web_exp

,client_id,visitor_id,visit_id,process_step,date_time,variation
0,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test
...,...,...,...,...,...,...
317230,9999729,834634258_21862004160,870243567_56915814033_814203,confirm,2017-05-08 16:09:40,Test
317231,9999729,604429154_69247391147,99583652_41711450505_426179,start,2017-04-05 13:40:49,Test
317232,9999729,604429154_69247391147,99583652_41711450505_426179,step_1,2017-04-05 13:41:04,Test
317233,9999832,145538019_54444341400,472154369_16714624241_585315,start,2017-05-16 16:46:03,Test


In [ ]:
#2. Vamos a crear una columna de Next step dentro de la misma visita y para el mismo client_id. 
df_merge_web_exp["next_step"] = (df_merge_web_exp.groupby(["client_id", "visit_id"])["process_step"].shift(-1))
df_merge_web_exp

,client_id,visitor_id,visit_id,process_step,date_time,variation,next_step
0,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,step_1
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test,step_2
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test,step_3
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test,confirm
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test,NaN
...,...,...,...,...,...,...,...
317230,9999729,834634258_21862004160,870243567_56915814033_814203,confirm,2017-05-08 16:09:40,Test,NaN
317231,9999729,604429154_69247391147,99583652_41711450505_426179,start,2017-04-05 13:40:49,Test,step_1
317232,9999729,604429154_69247391147,99583652_41711450505_426179,step_1,2017-04-05 13:41:04,Test,NaN
317233,9999832,145538019_54444341400,472154369_16714624241_585315,start,2017-05-16 16:46:03,Test,step_1


In [18]:
#3. Vamos a crear una columna de Next time dentro de la misma visita y para mismo client_id. 
df_merge_web_exp["next_time"] = (df_merge_web_exp.groupby(["client_id", "visit_id"])["date_time"].shift(-1))
df_merge_web_exp

,client_id,visitor_id,visit_id,process_step,date_time,variation,next_step,next_time
0,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,step_1,2017-04-15 12:58:03
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test,step_2,2017-04-15 12:58:35
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test,step_3,2017-04-15 13:00:14
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test,confirm,2017-04-15 13:00:34
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test,NaN,NaT
...,...,...,...,...,...,...,...,...
317230,9999729,834634258_21862004160,870243567_56915814033_814203,confirm,2017-05-08 16:09:40,Test,NaN,NaT
317231,9999729,604429154_69247391147,99583652_41711450505_426179,start,2017-04-05 13:40:49,Test,step_1,2017-04-05 13:41:04
317232,9999729,604429154_69247391147,99583652_41711450505_426179,step_1,2017-04-05 13:41:04,Test,NaN,NaT
317233,9999832,145538019_54444341400,472154369_16714624241_585315,start,2017-05-16 16:46:03,Test,step_1,2017-05-16 16:46:11


In [20]:
#4. Calcular tiempo entre pasos en segundos.
df_merge_web_exp["time_between__steps_seconds"] = (df_merge_web_exp["next_time"] - df_merge_web_exp["date_time"]).dt.total_seconds()
df_merge_web_exp

,client_id,visitor_id,visit_id,process_step,date_time,variation,next_step,next_time,time_in_seconds,time_between__steps_seconds
0,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,step_1,2017-04-15 12:58:03,7.0,7.0
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test,step_2,2017-04-15 12:58:35,32.0,32.0
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test,step_3,2017-04-15 13:00:14,99.0,99.0
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test,confirm,2017-04-15 13:00:34,20.0,20.0
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test,NaN,NaT,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
317230,9999729,834634258_21862004160,870243567_56915814033_814203,confirm,2017-05-08 16:09:40,Test,NaN,NaT,NaN,NaN
317231,9999729,604429154_69247391147,99583652_41711450505_426179,start,2017-04-05 13:40:49,Test,step_1,2017-04-05 13:41:04,15.0,15.0
317232,9999729,604429154_69247391147,99583652_41711450505_426179,step_1,2017-04-05 13:41:04,Test,NaN,NaT,NaN,NaN
317233,9999832,145538019_54444341400,472154369_16714624241_585315,start,2017-05-16 16:46:03,Test,step_1,2017-05-16 16:46:11,8.0,8.0


In [23]:
#5. Vamos a interpretar los tiempos entre un paso y el posterior medidos en segundos. 
df_merge_web_exp["time_between__steps_seconds"].describe().round(2)

count    247788.00
mean         83.86
std         214.05
min           0.00
25%          13.00
50%          36.00
75%          82.00
max       40235.00
Name: time_between__steps_seconds, dtype: float64

In [ ]:
#Comentarios: Vemos que el tiempo maximo es enorme, alreadedor de 11 horas. Lo cual nos indica que tenemos outliers. Vemos que esta variable tiene mucha variabilidad y la mayoria de los usuarios tarda bastante menos que la media, 83.86 segundos. 
#El tiempo maximo nos puede indicar que los usuarios pudieran dejar la sesion abierta durante horas en pausa, de ahi que nos salga un outlier tan alto. 

In [26]:
#6. Vamos a calcular ahora el KPI por grupo. 
time_mean_spent_each_step_groups = df_merge_web_exp.groupby("variation")["time_between__steps_seconds"].mean().round(2)
time_mean_spent_each_step_groups

variation
Control    83.52
Test       84.13
Name: time_between__steps_seconds, dtype: float64

In [ ]:
#Cometarios: Vemos que la media de tiempo por step es similar para ambos grupos, con una diferencia relativamente pequeña, de 6 segundos. 
#Sin embargo, tenemos que considerar los outliers que hemos identificado anteriormente. Por lo que debemos que analizar mas. 

In [27]:
#7. Vamos a calcular la mediana. 
time_median_spent_each_step_groups = df_merge_web_exp.groupby("variation")["time_between__steps_seconds"].median().round(2)
time_median_spent_each_step_groups

variation
Control    37.0
Test       35.0
Name: time_between__steps_seconds, dtype: float64

In [ ]:
#La mediana, en este caso, es mas representativa. Por lo que, podemos afirmar que la nueva interfaz es mejor que el modelo tradicional aunque no con gran diferencia. 
#Los usuarios del grupo test tardan menos en pasar de un step a otro, por lo que podemos interpretar que mas facil e intutivo terminar el proceso con la nueva interfaz.  

In [ ]:
#KPI 3: Error rate
#Calcular si hay un movimiento al step anterior para el mismo client_id y mismo visit_id. 
#Considerar que este el orden correcto: start, step_1, step_2, step_3, confirm. 


In [ ]:
#1. Vamos a crear una columna nueva error
df_merge_web_exp["error"] = (
    ((df_merge_web_exp["process_step"] == "step_1") & (df_merge_web_exp["next_step"] == "start")) |
    ((df_merge_web_exp["process_step"] == "step_2") & (df_merge_web_exp["next_step"] == "step_1")) |
    ((df_merge_web_exp["process_step"] == "step_3") & (df_merge_web_exp["next_step"] == "step_2")) |
    ((df_merge_web_exp["process_step"] == "confirm") & (df_merge_web_exp["next_step"] == "step_3"))
)
df_merge_web_exp

,client_id,visitor_id,visit_id,process_step,date_time,variation,next_step,next_time,time_in_seconds,time_between__steps_seconds,error
0,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,step_1,2017-04-15 12:58:03,7.0,7.0,False
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test,step_2,2017-04-15 12:58:35,32.0,32.0,False
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test,step_3,2017-04-15 13:00:14,99.0,99.0,False
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test,confirm,2017-04-15 13:00:34,20.0,20.0,False
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test,NaN,NaT,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...
317230,9999729,834634258_21862004160,870243567_56915814033_814203,confirm,2017-05-08 16:09:40,Test,NaN,NaT,NaN,NaN,False
317231,9999729,604429154_69247391147,99583652_41711450505_426179,start,2017-04-05 13:40:49,Test,step_1,2017-04-05 13:41:04,15.0,15.0,False
317232,9999729,604429154_69247391147,99583652_41711450505_426179,step_1,2017-04-05 13:41:04,Test,NaN,NaT,NaN,NaN,False
317233,9999832,145538019_54444341400,472154369_16714624241_585315,start,2017-05-16 16:46:03,Test,step_1,2017-05-16 16:46:11,8.0,8.0,False


In [31]:
#2. Vamos a calcular error_rate
error_rate = (df_merge_web_exp.groupby("variation")["error"].mean() * 100).round(2)
error_rate

variation
Control    4.51
Test       6.71
Name: error, dtype: float64

In [ ]:
#Comentarios: Aqui podemos ver que el grupo test presenta mayor ratio de error, lo cual nos indica que los usuarios exploran mas y/o corrigen pasos; pero, en relacion con los otros KPIs, completan mayor numero de usuarios y en menor tiempo. 

In [ ]:
#Based on the chosen KPIs, how does the new design's performance compare to the old one?
#En base a los KPIs calculados, podemos afirmar que la nueva interfaz parece funcionar mejor que el sistema tradicional, en términos generales. EL grupo TEST tuvo una mayor completion rate, lo cual indica que hubo un mayor numero de usuarios que completaron el proceso con la nueva interfaz. Además, la mediana del tiempo entre pasos es inferior con respecto al grupo Cntrol, lo cual sugiere una experiencia más eificiente para el usuario. Aunque, debemos mencionar que error_rate del grupo Test fue más alto, lo cual nos plantea una posible confusión por parte del usuario. 
#En general, podemos afirmar que el nuevo diseño parece factible, aunque con ciertas mejoras. 